# 03 — Build Your Own MCP Server

**Goal:** Implement a minimal but real MCP server — initialization handshake, tool discovery, and tool execution.

## MCP Lifecycle

```
Client                              Server
  │                                    │
  │──── initialize (request) ─────────→│  "Here's who I am, what I support"
  │←─── initialize (response) ─────────│  "Here's who I am, what I support"
  │──── notifications/initialized ────→│  "OK, let's go" (no response)
  │                                    │
  │──── tools/list (request) ─────────→│  "What tools do you have?"
  │←─── tools/list (response) ─────────│  "Here's the list"
  │                                    │
  │──── tools/call (request) ─────────→│  "Run this tool"
  │←─── tools/call (response) ─────────│  "Here's the result"
  │                                    │
  │──── [close stdin] ────────────────→│  Shutdown
```

That's it. Three methods + one notification = a working MCP server.

## Exercise 3.1 — Define your tools

MCP tools have three parts:
1. **name** — identifier
2. **description** — what the tool does (the LLM reads this to decide when to use it)
3. **inputSchema** — JSON Schema describing parameters

Define 3 tools: `add`, `get_weather` (fake), and `list_files`.

In [ ]:
import json
import os

# Tool definitions (what gets returned by tools/list)
TOOL_DEFINITIONS = [
    {
        "name": "add",
        "description": "Add two numbers together",
        "inputSchema": {
            "type": "object",
            "properties": {
                "a": {"type": "number", "description": "First number"},
                "b": {"type": "number", "description": "Second number"},
            },
            "required": ["a", "b"],
        },
    },
    # TODO: Define get_weather tool
    # - name: "get_weather"
    # - description: something useful for an LLM
    # - inputSchema: one required param "city" (string)
    
    # TODO: Define list_files tool
    # - name: "list_files"
    # - description: list files in a directory
    # - inputSchema: one optional param "path" (string, default ".")
]

print(f"Defined {len(TOOL_DEFINITIONS)} tools:")
for t in TOOL_DEFINITIONS:
    print(f"  - {t['name']}: {t['description']}")

assert len(TOOL_DEFINITIONS) == 3, "Define all 3 tools!"

## Exercise 3.2 — Implement tool handlers

Each tool needs an actual Python function that runs when called.

In [ ]:
def tool_add(arguments: dict) -> list[dict]:
    """MCP tools return a list of content blocks."""
    result = arguments["a"] + arguments["b"]
    return [{"type": "text", "text": str(result)}]

def tool_get_weather(arguments: dict) -> list[dict]:
    # TODO: return fake weather for the given city
    # Format: [{"type": "text", "text": "..."}]
    pass

def tool_list_files(arguments: dict) -> list[dict]:
    # TODO: list files in the given path (default ".")
    # Use os.listdir()
    # Format: [{"type": "text", "text": "file1\nfile2\nfile3"}]
    pass


TOOL_HANDLERS = {
    "add": tool_add,
    "get_weather": tool_get_weather,
    "list_files": tool_list_files,
}

# Test
print(tool_add({"a": 3, "b": 4}))
print(tool_get_weather({"city": "Hanoi"}))
print(tool_list_files({"path": "."}))

## Exercise 3.3 — Build the MCP server class

Now combine everything: handle the initialization handshake, tool discovery, and tool calling.

This is the core of MCP. Get this right and you understand the entire protocol.

In [ ]:
class MiniMCPServer:
    """A minimal MCP server that handles the core protocol."""
    
    PROTOCOL_VERSION = "2024-11-05"
    
    def __init__(self, name: str, tools: list[dict], handlers: dict):
        self.name = name
        self.tools = tools
        self.handlers = handlers
        self.initialized = False
    
    def handle_message(self, msg: dict) -> dict | None:
        """Process one JSON-RPC message. Returns response or None for notifications."""
        method = msg.get("method", "")
        msg_id = msg.get("id")
        params = msg.get("params", {})
        
        # --- Notifications (no id → no response) ---
        if msg_id is None:
            if method == "notifications/initialized":
                self.initialized = True
            return None
        
        # --- Requests (has id → must respond) ---
        
        if method == "initialize":
            # TODO: return a response with:
            # - protocolVersion: self.PROTOCOL_VERSION
            # - capabilities: {"tools": {}}  (we support tools)
            # - serverInfo: {"name": self.name, "version": "1.0.0"}
            pass
        
        elif method == "tools/list":
            # TODO: return a response with:
            # - result: {"tools": self.tools}
            pass
        
        elif method == "tools/call":
            # TODO:
            # 1. Get tool name from params["name"]
            # 2. Get arguments from params["arguments"]
            # 3. Look up handler in self.handlers
            # 4. If not found, return error response (code -32602)
            # 5. Call handler, return result: {"content": [...], "isError": false}
            # 6. If handler raises, return result: {"content": [{"type": "text", "text": str(e)}], "isError": true}
            #    NOTE: tool errors are NOT JSON-RPC errors! They're successful responses with isError=true
            pass
        
        else:
            return {"jsonrpc": "2.0", "id": msg_id, "error": {"code": -32601, "message": f"Unknown method: {method}"}}
    
    def _response(self, msg_id: int, result: dict) -> dict:
        """Helper to build a success response."""
        return {"jsonrpc": "2.0", "id": msg_id, "result": result}

## Exercise 3.4 — Test the server (in-process)

Before running as a subprocess, test it directly in Python.

In [ ]:
server = MiniMCPServer("test-server", TOOL_DEFINITIONS, TOOL_HANDLERS)

# Step 1: Initialize handshake
resp = server.handle_message({
    "jsonrpc": "2.0", "id": 1, "method": "initialize",
    "params": {
        "protocolVersion": "2024-11-05",
        "capabilities": {},
        "clientInfo": {"name": "test-client", "version": "1.0.0"}
    }
})
print("1. Initialize response:")
print(json.dumps(resp, indent=2))
assert resp["result"]["serverInfo"]["name"] == "test-server"
assert resp["result"]["capabilities"]["tools"] == {}

# Step 2: Send initialized notification
resp = server.handle_message({"jsonrpc": "2.0", "method": "notifications/initialized"})
print("\n2. Initialized notification:", resp)  # should be None
assert resp is None
assert server.initialized == True

# Step 3: List tools
resp = server.handle_message({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})
print("\n3. Tools list:")
for tool in resp["result"]["tools"]:
    print(f"  - {tool['name']}: {tool['description']}")
assert len(resp["result"]["tools"]) == 3

# Step 4: Call a tool
resp = server.handle_message({
    "jsonrpc": "2.0", "id": 3, "method": "tools/call",
    "params": {"name": "add", "arguments": {"a": 10, "b": 32}}
})
print("\n4. Tool call (add):")
print(json.dumps(resp, indent=2))
assert resp["result"]["content"][0]["text"] == "42"
assert resp["result"]["isError"] == False

# Step 5: Call weather tool
resp = server.handle_message({
    "jsonrpc": "2.0", "id": 4, "method": "tools/call",
    "params": {"name": "get_weather", "arguments": {"city": "Hanoi"}}
})
print("\n5. Tool call (weather):")
print(json.dumps(resp, indent=2))

# Step 6: Call unknown tool
resp = server.handle_message({
    "jsonrpc": "2.0", "id": 5, "method": "tools/call",
    "params": {"name": "nonexistent", "arguments": {}}
})
print("\n6. Unknown tool:")
print(json.dumps(resp, indent=2))

print("\n✓ Full MCP handshake + tool call working!")

### Question 3.4

Look at step 6 (unknown tool). Did you return it as:
- A) A JSON-RPC error (`"error": {"code": ...}`)
- B) A tool error (`"result": {"content": [...], "isError": true}`)

MCP spec says tool-not-found should be a JSON-RPC error (option A), but a tool that *runs* and *fails* (e.g., file not found) should be option B. Why the distinction?

**Hint:** Think about what the LLM sees. A JSON-RPC error means "the protocol broke." A tool error means "I tried but it didn't work" — the LLM can try a different approach.

*Your answer:*


## Exercise 3.5 — Write the server as a standalone script

Combine `MiniMCPServer` with the stdio transport from notebook 02.

In [ ]:
server_code = '''\
#!/usr/bin/env python3
"""Mini MCP Server — implements initialize + tools/list + tools/call over stdio."""
import json
import os
import sys

def log(msg):
    print(f"[mcp-server] {msg}", file=sys.stderr, flush=True)

# --- Tool definitions ---
TOOLS = [
    {
        "name": "add",
        "description": "Add two numbers together",
        "inputSchema": {
            "type": "object",
            "properties": {
                "a": {"type": "number", "description": "First number"},
                "b": {"type": "number", "description": "Second number"},
            },
            "required": ["a", "b"],
        },
    },
    {
        "name": "get_weather",
        "description": "Get current weather for a city (fake data for demo)",
        "inputSchema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"},
            },
            "required": ["city"],
        },
    },
    {
        "name": "list_files",
        "description": "List files in a directory",
        "inputSchema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Directory path (default: current dir)"},
            },
        },
    },
]

# --- Tool handlers ---
def tool_add(args):
    return [{"type": "text", "text": str(args["a"] + args["b"])}]

def tool_get_weather(args):
    city = args["city"]
    return [{"type": "text", "text": f"{city}: 28°C, 80% humidity, partly cloudy"}]

def tool_list_files(args):
    path = args.get("path", ".")
    try:
        files = os.listdir(path)
        return [{"type": "text", "text": "\\n".join(sorted(files))}]
    except OSError as e:
        raise ValueError(f"Cannot list {path}: {e}")

HANDLERS = {"add": tool_add, "get_weather": tool_get_weather, "list_files": tool_list_files}

PROTOCOL_VERSION = "2024-11-05"

# --- Message handling ---
def handle(msg: dict) -> dict | None:
    method = msg.get("method", "")
    msg_id = msg.get("id")
    params = msg.get("params", {})
    
    if msg_id is None:  # notification
        log(f"notification: {method}")
        return None
    
    if method == "initialize":
        return {"jsonrpc": "2.0", "id": msg_id, "result": {
            "protocolVersion": PROTOCOL_VERSION,
            "capabilities": {"tools": {}},
            "serverInfo": {"name": "mini-mcp", "version": "1.0.0"},
        }}
    
    if method == "tools/list":
        return {"jsonrpc": "2.0", "id": msg_id, "result": {"tools": TOOLS}}
    
    if method == "tools/call":
        name = params.get("name", "")
        arguments = params.get("arguments", {})
        handler = HANDLERS.get(name)
        if not handler:
            return {"jsonrpc": "2.0", "id": msg_id, "error": {"code": -32602, "message": f"Unknown tool: {name}"}}
        try:
            content = handler(arguments)
            return {"jsonrpc": "2.0", "id": msg_id, "result": {"content": content, "isError": False}}
        except Exception as e:
            return {"jsonrpc": "2.0", "id": msg_id, "result": {"content": [{"type": "text", "text": str(e)}], "isError": True}}
    
    return {"jsonrpc": "2.0", "id": msg_id, "error": {"code": -32601, "message": f"Unknown method: {method}"}}

# --- Main loop ---
def main():
    log("Mini MCP Server started")
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        log(f"← {line[:100]}")
        resp = handle(json.loads(line))
        if resp:
            out = json.dumps(resp)
            print(out, flush=True)
            log(f"→ {out[:100]}")
    log("Server shutting down")

if __name__ == "__main__":
    main()
'''

with open("mini_mcp_server.py", "w") as f:
    f.write(server_code)
print("Created mini_mcp_server.py")
print("You can test it manually: echo '{\"jsonrpc\":\"2.0\",\"id\":1,\"method\":\"initialize\",\"params\":{}}' | python3 mini_mcp_server.py")

## Summary

You just built an MCP server from scratch. It handles:

| Method | Purpose |
|--------|---------|
| `initialize` | Handshake — exchange capabilities |
| `notifications/initialized` | Client confirms ready |
| `tools/list` | Tool discovery |
| `tools/call` | Tool execution |

**That's 90% of what real MCP servers do.** The rest (resources, prompts, sampling) follows the same pattern — just different method names.

**Next notebook:** Build the MCP client that connects to this server